[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/heitorramos/icd/blob/main/exemplos/19-verossimilhanca-regressao/notebook-colab.ipynb)


In [ ]:
# Preparação automática para execução no Google Colab.
# Fora do Colab, esta célula não altera o diretório de trabalho.
try:
    import google.colab  # type: ignore
except ImportError:
    pass
else:
    import os
    import subprocess
    from pathlib import Path

    repository = Path("/content/icd")
    if not repository.exists():
        subprocess.run([
            "git", "clone", "--depth", "1",
            "https://github.com/heitorramos/icd.git", str(repository)
        ], check=True)
    os.chdir(repository / "exemplos/19-verossimilhanca-regressao")
    print("Material preparado em:", Path.cwd())


# Verossimilhança gaussiana e regressão linear

Material de apoio — Aula 19

## Objetivos

Este guia formula a regressão como modelo probabilístico, deriva sua
log-verossimilhança e mostra por que máxima verossimilhança gaussiana e
mínimos quadrados estimam os mesmos coeficientes. A discussão final
esclarece o papel real da normalidade.

## Como estudar este capítulo

Na Aula 17, a reta foi escolhida por minimizar a soma dos resíduos ao
quadrado. Agora perguntamos: **qual modelo probabilístico conduz
exatamente ao mesmo critério?** A resposta aparece quando supomos que,
para cada valor dos preditores, a resposta oscila ao redor da reta
segundo uma distribuição Normal com variância constante.

O capítulo separa duas camadas que muitas vezes são misturadas. A
primeira é a **média condicional**, descrita pela reta. A segunda é a
**distribuição das observações ao redor dessa média**, descrita pelo
termo de erro. A normalidade pertence à segunda camada. Ela não é
necessária para desenhar a reta de mínimos quadrados, mas sustenta a
forma específica da verossimilhança e os procedimentos inferenciais
usuais em amostras pequenas.

Ao acompanhar as equações, identifique sempre o papel de cada termo:
$y_i$ é a resposta observada, $x_i^T\beta$ é a média prevista,
$y_i-x_i^T\beta$ é o resíduo correspondente e $\sigma^2$ controla a
dispersão ao redor da reta.

## Base de dados de apoio

Continuamos com [Medical Insurance
Cost](https://www.kaggle.com/datasets/mosapabdelghany/medical-insurance-cost-dataset).
A resposta é `charges`, o preditor é `age` e `smoker` será usado no
diagnóstico.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

sns.set_theme(style="whitegrid", context="notebook")
df = pd.read_csv(Path("../16-correlacao/data/insurance.csv"))
df[["age", "charges", "smoker"]].head()

> **Interpretação**
>
> Cada linha é uma pessoa. A regressão simples descreve despesas médias
> por idade; ela não incorpora todas as características que distinguem
> indivíduos com a mesma idade.

## Modelo para a média condicional

$$Y_i=\beta_0+\beta_1x_i+\varepsilon_i,
\qquad E[\varepsilon_i\mid X_i=x_i]=0.$$

$Y_i$ é a resposta, $x_i$ o preditor, $\beta_0$ o intercepto, $\beta_1$
a inclinação e $\varepsilon_i$ o erro populacional. A condição de média
zero implica

$$E[Y_i\mid X_i=x_i]=\beta_0+\beta_1x_i.$$

## Acrescentando uma distribuição

Para construir a verossimilhança gaussiana, assumimos

$$\varepsilon_i\overset{iid}{\sim}N(0,\sigma^2).$$

Equivalentemente,

$$Y_i\mid X_i=x_i\sim N(\beta_0+\beta_1x_i,\sigma^2).$$

A média depende de $x_i$ e a variância condicional é constante.

## Densidade condicional

$$
f(y_i\mid x_i;\beta_0,\beta_1,\sigma^2)
=\frac{1}{\sqrt{2\pi\sigma^2}}
\exp\left[-\frac{(y_i-\beta_0-\beta_1x_i)^2}{2\sigma^2}\right].
$$

O termo $y_i-\beta_0-\beta_1x_i$ é o erro vertical relativo a uma reta
candidata. Erros maiores recebem densidade menor.

## Verossimilhança conjunta

Sob independência condicional,

$$
L(\beta_0,\beta_1,\sigma^2)
=\prod_{i=1}^n f(y_i\mid x_i;\beta_0,\beta_1,\sigma^2).
$$

Tomando log:

$$
\ell=-\frac n2\log(2\pi)-\frac n2\log(\sigma^2)
-\frac{1}{2\sigma^2}\sum_i(y_i-\beta_0-\beta_1x_i)^2.
$$

## Equivalência com mínimos quadrados

Para $\sigma^2>0$ fixo, os dois primeiros termos não dependem dos
coeficientes. Portanto,

$$
\arg\max_{\beta_0,\beta_1}\ell
=\arg\min_{\beta_0,\beta_1}
\sum_i(y_i-\beta_0-\beta_1x_i)^2.
$$

> **Interpretação**
>
> OLS e máxima verossimilhança gaussiana não são dois algoritmos que
> coincidiram por acaso. O log negativo da densidade normal é uma
> constante mais o erro quadrático escalado.

## Cálculo manual dos coeficientes

In [ ]:
x = df["age"].to_numpy(float)
y = df["charges"].to_numpy(float)
xbar, ybar = x.mean(), y.mean()
b1 = np.sum((x-xbar)*(y-ybar))/np.sum((x-xbar)**2)
b0 = ybar-b1*xbar
yhat = b0+b1*x
resid = y-yhat
pd.Series({"beta0": b0, "beta1": b1}).round(4)

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x="age", y="charges", hue="smoker", alpha=.4)
order = np.argsort(x)
plt.plot(x[order], yhat[order], color="darkorange", linewidth=3)
plt.show()

> **Interpretação**
>
> Um ano adicional de idade está associado a aproximadamente US\$ 257,72
> na despesa média prevista. O grande espalhamento vertical mostra que
> idade isolada produz previsões individuais frágeis.

## Verificação numérica da equivalência

In [ ]:
sigma2 = np.mean(resid**2)

def sse(beta0, beta1):
    return np.sum((y-beta0-beta1*x)**2)

def loglik(beta0, beta1, sigma2):
    return (-len(y)/2*np.log(2*np.pi)
            -len(y)/2*np.log(sigma2)
            -sse(beta0, beta1)/(2*sigma2))

candidates = pd.DataFrame({
    "modelo": ["inclinação menor", "OLS/MLE", "inclinação maior"],
    "beta1": [.5*b1, b1, 1.5*b1],
})
candidates["beta0"] = ybar-candidates.beta1*xbar
candidates["SSE"] = [sse(a,b) for a,b in zip(candidates.beta0,candidates.beta1)]
candidates["logLik"] = [loglik(a,b,sigma2) for a,b in zip(candidates.beta0,candidates.beta1)]
candidates.round(2)

> **Interpretação**
>
> O candidato com menor SSE é exatamente o de maior log-verossimilhança.
> Alterar a inclinação enquanto a reta passa pelo centro dos dados piora
> ambos os critérios em direções opostas.

## Estimação da variância

Depois de estimar $\beta_0$ e $\beta_1$, os resíduos são

$$
e_i=y_i-\widehat y_i
   =y_i-\widehat\beta_0-\widehat\beta_1x_i,
$$

e sua soma de quadrados é

$$
\operatorname{SSE}=\sum_{i=1}^n e_i^2.
$$

Na log-verossimilhança gaussiana, os termos que ainda dependem da
variância são

$$
\ell(\sigma^2)
=-\frac n2\log(2\pi)
-\frac n2\log(\sigma^2)
-\frac{\operatorname{SSE}}{2\sigma^2}.
$$

Aqui, $n$ é o número de observações e $\sigma^2$ é a variância
populacional dos erros ao redor da reta. Para deixar a derivação mais
legível, escreva $v=\sigma^2$. Então,

$$
\ell(v)
=-\frac n2\log(2\pi)
-\frac n2\log v
-\frac{\operatorname{SSE}}{2v}.
$$

Derivando em relação a $v$:

$$
\frac{\partial\ell}{\partial v}
=-\frac{n}{2v}
+\frac{\operatorname{SSE}}{2v^2}.
$$

O segundo termo tem sinal positivo porque

$$
\frac{d}{dv}\left(-\frac{\operatorname{SSE}}{2}v^{-1}\right)
=\frac{\operatorname{SSE}}{2}v^{-2}.
$$

No máximo da verossimilhança, a derivada é zero. Assim,

$$
-\frac{n}{2v}
+\frac{\operatorname{SSE}}{2v^2}=0.
$$

Multiplicando os dois lados por $2v^2$:

$$
-nv+\operatorname{SSE}=0,
$$

e, portanto,

$$
\widehat v=\frac{\operatorname{SSE}}{n}.
$$

Como $v=\sigma^2$, obtemos

$$
\boxed{
\widehat\sigma^2_{MV}
=\frac{\operatorname{SSE}}{n}
=\frac1n\sum_{i=1}^n(y_i-\widehat y_i)^2
}.
$$

A segunda derivada avaliada nesse ponto é negativa:

$$
\left.
\frac{\partial^2\ell}{\partial v^2}
\right|_{v=\widehat v}
=-\frac{n}{2\widehat v^2}<0,
$$

confirmando que o ponto encontrado é um máximo. Intuitivamente, a máxima
verossimilhança escolhe como variância a média dos resíduos quadráticos.

In [ ]:
sigma2_mle = np.mean(resid**2)
sigma2_unbiased = np.sum(resid**2)/(len(y)-2)
pd.Series({
    "sigma_MLE": np.sqrt(sigma2_mle),
    "sigma_corrigido": np.sqrt(sigma2_unbiased),
}).round(2)

> **Interpretação**
>
> Dividir por $n$ é o resultado da maximização direta da
> verossimilhança. Esse estimador, porém, tende a subestimar a variância
> em amostras finitas, pois a mesma amostra foi usada para ajustar
> $\beta_0$ e $\beta_1$. Como dois coeficientes foram estimados, restam
> $n-2$ graus de liberdade residuais, e a estimativa não viesada usual é
>
> $$
> s^2=\frac{\operatorname{SSE}}{n-2}.
> $$
>
> Portanto, “máxima verossimilhança” e “não viesado” são propriedades
> diferentes: $\operatorname{SSE}/n$ maximiza a verossimilhança,
> enquanto $\operatorname{SSE}/(n-2)$ aplica a correção de graus de
> liberdade.

## Os resíduos precisam ser normais?

Não para calcular OLS. A minimização da SSE é uma operação algébrica que
não exige distribuição. Para que OLS estime a média condicional sem viés
sistemático, a condição essencial é $E[\varepsilon\mid X]=0$.

Normalidade é usada para declarar a verossimilhança gaussiana correta e
obter certas distribuições exatas de estatísticas em amostras finitas.

## Erros e resíduos

-   $\varepsilon_i$ é o erro populacional não observado;
-   $e_i=y_i-\widehat y_i$ é o resíduo após estimar a reta;
-   resíduos somam zero quando há intercepto;
-   resíduos não são independentes entre si;
-   sua distribuição serve como aproximação diagnóstica à distribuição
    dos erros.

## Consequências de violações

| hipótese | se falhar |
|------------------------------------|------------------------------------|
| média linear correta | coeficientes podem resumir uma forma inadequada |
| $E[\varepsilon\mid X]=0$ | associação pode estar enviesada por estrutura omitida |
| variância constante | erro-padrão clássico pode estar errado |
| independência | produto da verossimilhança e incerteza usual falham |
| normalidade | inferência exata gaussiana e previsão normal podem falhar |

## Outras distribuições, outras perdas

Para a Normal,

$$-\log f(e)=C+e^2/(2\sigma^2).$$

Por isso aparece a perda quadrática. Erros Laplace produzem perda
absoluta; Bernoulli produz log-loss. Escolher uma distribuição é também
escolher como discrepâncias serão penalizadas.

## Exemplo completo: da pergunta ao diagnóstico

Vamos reunir as etapas da técnica em uma única análise. A pergunta é:

> Qual é a associação média entre idade e despesas médicas nesta
> amostra?

O modelo de regressão linear simples é

$$
Y_i=\beta_0+\beta_1\,age_i+\varepsilon_i,
\qquad
\varepsilon_i\overset{iid}{\sim}N(0,\sigma^2).
$$

Neste modelo, $Y_i$ é a despesa anual da pessoa $i$ e $\beta_1$ é a
variação da despesa média associada a um ano adicional de idade. A
formulação é associacional: sem um desenho causal, $\beta_1$ não
representa o efeito de envelhecer uma pessoa.

### Etapa 1: ajuste do modelo

In [ ]:
X = sm.add_constant(df["age"])
ajuste = sm.OLS(df["charges"], X).fit()

tabela_coeficientes = pd.DataFrame({
    "estimativa": ajuste.params,
    "erro_padrão": ajuste.bse,
    "estatística_t": ajuste.tvalues,
    "valor_p": ajuste.pvalues,
})
tabela_coeficientes.round(4)

In [ ]:
x_grid = np.linspace(df["age"].min(), df["age"].max(), 200)
X_grid = sm.add_constant(x_grid)
pred = ajuste.get_prediction(X_grid).summary_frame(alpha=.05)

plt.figure(figsize=(9, 5))
plt.scatter(df["age"], df["charges"], alpha=.20, s=18, label="observações")
plt.fill_between(
    x_grid,
    pred["mean_ci_lower"].to_numpy(),
    pred["mean_ci_upper"].to_numpy(),
    alpha=.25,
    color="darkorange",
    label="IC 95% para a média",
)
plt.plot(x_grid, pred["mean"], color="darkorange", linewidth=3, label="reta estimada")
plt.xlabel("idade (anos)")
plt.ylabel("despesas médicas (US$)")
plt.legend()
plt.show()

O ajuste fornece

$$
\widehat{charges}=3165{,}89+257{,}72\,age.
$$

> **Interpretação**
>
> Segundo o modelo, um ano adicional de idade está associado a US\$
> 257,72 a mais na despesa média prevista. Essa interpretação é sobre a
> média condicional, não sobre o gasto exato de uma pessoa. O valor
> $R^2=0{,}089$ mostra que idade sozinha explica apenas 8,9% da
> variabilidade observada.

### Etapa 2: intervalo de confiança para $\beta_1$

O erro-padrão de $\widehat\beta_1$ mede quanto a inclinação estimada
variaria se repetíssemos o processo de amostragem. Para derivá-lo,
começamos pela fórmula de mínimos quadrados:

$$
\widehat\beta_1
=\frac{\sum_{i=1}^n(x_i-\bar x)(y_i-\bar y)}{S_{xx}},
\qquad
S_{xx}=\sum_{i=1}^n(x_i-\bar x)^2.
$$

Substitua o modelo $Y_i=\beta_0+\beta_1x_i+\varepsilon_i$. Como

$$
Y_i-\bar Y
=\beta_1(x_i-\bar x)+(\varepsilon_i-\bar\varepsilon),
$$

temos

$$
\widehat\beta_1
=\beta_1+
\frac{\sum_i(x_i-\bar x)(\varepsilon_i-\bar\varepsilon)}{S_{xx}}.
$$

Como $\sum_i(x_i-\bar x)=0$, o termo que multiplica $\bar\varepsilon$
desaparece:

$$
\widehat\beta_1
=\beta_1+
\frac{\sum_i(x_i-\bar x)\varepsilon_i}{S_{xx}}.
$$

Essa expressão mostra que $\widehat\beta_1$ é o valor populacional
$\beta_1$ mais uma combinação dos erros aleatórios. Condicionalmente aos
valores de $X$, assumimos erros independentes com

$$
E[\varepsilon_i\mid X]=0,
\qquad
\operatorname{Var}(\varepsilon_i\mid X)=\sigma^2.
$$

Portanto,

$$
\begin{aligned}
\operatorname{Var}(\widehat\beta_1\mid X)
&=\operatorname{Var}\left[
\frac{\sum_i(x_i-\bar x)\varepsilon_i}{S_{xx}}
\,\middle|\,X\right]\\
&=\frac{1}{S_{xx}^2}
\sum_i(x_i-\bar x)^2\sigma^2\\
&=\frac{\sigma^2}{S_{xx}}.
\end{aligned}
$$

Na segunda linha, usamos a independência dos erros: a variância da soma
é a soma das variâncias. Na última linha, reconhecemos que
$\sum_i(x_i-\bar x)^2=S_{xx}$. Assim, o desvio-padrão teórico da
inclinação é

$$
\operatorname{SD}(\widehat\beta_1\mid X)
=\frac{\sigma}{\sqrt{S_{xx}}}.
$$

Como $\sigma$ é desconhecido, substituímos por

$$
s=\sqrt{\frac{\operatorname{SSE}}{n-2}}.
$$

Obtemos, finalmente, o erro-padrão estimado:

$$
\boxed{
\operatorname{SE}(\widehat\beta_1)
=\frac{s}{\sqrt{S_{xx}}}
=\sqrt{\frac{s^2}{S_{xx}}}
}.
$$

O denominador $n-2$ aparece porque usamos a amostra para estimar dois
coeficientes, $\beta_0$ e $\beta_1$. No exemplo da base de seguros:

In [ ]:
Sxx_exemplo = np.sum((x - x.mean())**2)
SSE_exemplo = np.sum(ajuste.resid**2)
s2_exemplo = SSE_exemplo / (len(x) - 2)
s_exemplo = np.sqrt(s2_exemplo)
se_beta1_manual = s_exemplo / np.sqrt(Sxx_exemplo)

pd.Series({
    "Sxx (anos²)": Sxx_exemplo,
    "SSE (US$²)": SSE_exemplo,
    "s (US$)": s_exemplo,
    "SE(beta1) (US$/ano)": se_beta1_manual,
}).round(2)

Numericamente,

$$
\operatorname{SE}(\widehat\beta_1)
=\frac{11.560{,}31}{\sqrt{263.925{,}65}}
=\frac{11.560{,}31}{513{,}74}
=22{,}50\text{ US\$/ano}.
$$

> **Como interpretar a fórmula**
>
> -   Mais dispersão vertical dos dados aumenta $s$ e torna a inclinação
>     menos precisa.
> -   Maior dispersão das idades aumenta $S_{xx}$ e torna a inclinação
>     mais precisa.
> -   Se todas as idades fossem iguais, $S_{xx}=0$ e não seria possível
>     estimar uma inclinação.

Com o erro-padrão calculado, o intervalo bilateral de nível $1-\alpha$ é

$$
\widehat\beta_1
\pm
t_{1-\alpha/2,n-2}
\operatorname{SE}(\widehat\beta_1).
$$

In [ ]:
ic_beta1 = ajuste.conf_int(alpha=.05).loc["age"]
pd.Series({
    "beta1": ajuste.params["age"],
    "erro_padrão": ajuste.bse["age"],
    "limite_inferior_95%": ic_beta1.iloc[0],
    "limite_superior_95%": ic_beta1.iloc[1],
}).round(2)

Numericamente,

$$
257{,}72\pm1{,}962(22{,}50)
=[213{,}58;\ 301{,}87].
$$

> **Interpretação**
>
> Sob as hipóteses do modelo, o intervalo de confiança de 95% para a
> variação da despesa média associada a um ano de idade vai de US\$
> 213,58 a US\$ 301,87. O intervalo quantifica incerteza sobre a
> inclinação populacional; ele não é um intervalo para a despesa de uma
> pessoa específica.

### Etapa 3: teste de hipótese para $\beta_1$

Para verificar se há evidência de uma associação linear, testamos

$$
H_0:\beta_1=0
\qquad\text{contra}\qquad
H_A:\beta_1\ne0.
$$

Sob $H_0$ e as hipóteses do modelo, usamos

$$
T=\frac{\widehat\beta_1-0}
{\operatorname{SE}(\widehat\beta_1)}
\sim t_{n-2}.
$$

Se a variância populacional $\sigma^2$ fosse conhecida, a padronização
usaria $\sigma$ e teria distribuição Normal padrão. Na prática,
substituímos $\sigma$ por sua estimativa $s$; essa incerteza adicional
produz a distribuição $t$. À medida que os graus de liberdade aumentam,
a distribuição $t$ converge para a Normal. Com $n-2=1.336$ graus de
liberdade, as duas referências são quase indistinguíveis, mas o cálculo
clássico correto continua sendo o teste $t$.

In [ ]:
pd.Series({
    "estatística_t": ajuste.tvalues["age"],
    "graus_de_liberdade": int(ajuste.df_resid),
    "valor_p_bilateral": ajuste.pvalues["age"],
})

O resultado é

$$
t=11{,}45,
\qquad gl=1.336,
\qquad p=4{,}89\times10^{-29}.
$$

> **Interpretação**
>
> Ao nível de 5%, rejeitamos $H_0$. Os dados fornecem forte evidência de
> que a inclinação não é zero. Isso não significa que a probabilidade de
> $H_0$ seja o valor-$p$, não mede a importância prática da associação e
> não demonstra causalidade.

#### Uma segunda rota: teste bootstrap

Também podemos aproximar a distribuição amostral da inclinação sem impor
uma forma Normal aos erros. No **bootstrap por pares**, reamostramos com
reposição as linhas $(x_i,y_i)$ e ajustamos uma nova regressão em cada
réplica:

$$
\widehat\beta_1^{*(1)},\ldots,\widehat\beta_1^{*(B)}.
$$

Essa distribuição bootstrap reproduz a variação ao redor da estimativa
observada $\widehat\beta_1$, e não ao redor do valor nulo $0$. Portanto,
para testar $H_0:\beta_1=0$, recentralizamos as réplicas:

$$
D_b^*=\widehat\beta_1^{*(b)}-\widehat\beta_1.
$$

Sob a hipótese nula, $D_b^*$ aproxima a flutuação de uma inclinação
estimada em torno de zero. A distância observada em relação a $H_0$ é

$$
D_{obs}=\widehat\beta_1-0.
$$

Para um teste bilateral, estimamos

$$
\widehat p_{boot}
=\frac{1+\sum_{b=1}^{B}
I\!\left(|D_b^*|\ge |D_{obs}|\right)}{B+1}.
$$

Na expressão, $I(\cdot)$ vale 1 quando a condição é verdadeira e 0 caso
contrário; $B$ é o número de réplicas. A correção $+1$ evita reportar
valor-$p$ exatamente igual a zero por limitação da simulação.

In [ ]:
rng_boot = np.random.default_rng(1919)
B = 20_000
n = len(df)
boot_beta1 = []

# O processamento em lotes evita manter todas as 20 mil amostras na memória.
for inicio in range(0, B, 500):
    tamanho_lote = min(500, B - inicio)
    indices = rng_boot.integers(0, n, size=(tamanho_lote, n))
    xb = x[indices]
    yb = y[indices]
    xb_c = xb - xb.mean(axis=1, keepdims=True)
    yb_c = yb - yb.mean(axis=1, keepdims=True)
    inclinacoes = np.sum(xb_c * yb_c, axis=1) / np.sum(xb_c**2, axis=1)
    boot_beta1.append(inclinacoes)

boot_beta1 = np.concatenate(boot_beta1)
desvios_nulos = boot_beta1 - ajuste.params["age"]
distancia_observada = abs(ajuste.params["age"])
extremos = np.sum(np.abs(desvios_nulos) >= distancia_observada)
p_boot = (extremos + 1) / (B + 1)

pd.Series({
    "beta1_observado": ajuste.params["age"],
    "erro_padrão_bootstrap": boot_beta1.std(ddof=1),
    "IC_percentil_2,5%": np.quantile(boot_beta1, .025),
    "IC_percentil_97,5%": np.quantile(boot_beta1, .975),
    "réplicas_tão_extremas": extremos,
    "valor_p_bootstrap": p_boot,
})

> **Interpretação**
>
> O erro-padrão bootstrap é US\$ 22,46 e o intervalo percentil de 95% é
> $[213{,}14;300{,}76]$, muito próximos dos resultados teóricos. Nenhuma
> das 20.000 réplicas recentralizadas foi tão extrema quanto a
> inclinação observada; com a correção $+1$, obtemos
> $\widehat p_{boot}\approx5{,}0\times10^{-5}$.
>
> Isso confirma a rejeição de $H_0$, mas não reproduz numericamente o
> valor-$p$ teórico de ordem $10^{-29}$. Com $B$ réplicas, a menor
> resolução simulada é aproximadamente $1/B$. O bootstrap reduz a
> dependência da hipótese de normalidade, mas não corrige amostragem não
> representativa, dependência entre linhas nem uma função média mal
> especificada.

Uma alternativa é ajustar o modelo sob $H_0$, recentralizar os resíduos
e reamostrá-los. Essa versão é chamada **bootstrap residual sob a
hipótese nula**. O bootstrap por pares usado aqui é mais direto e
preserva a distribuição empírica conjunta de idade e despesas.

#### Quando usar o teste $t$ e quando usar bootstrap?

As duas técnicas aproximam a incerteza de maneiras diferentes. O teste
$t$ parte de uma distribuição deduzida matematicamente do modelo; o
bootstrap constrói uma distribuição empírica por reamostragem. Nenhuma
das duas rotas é automaticamente superior em todas as situações.

| aspecto | teste $t$ clássico | bootstrap por pares |
|------------------------|------------------------|------------------------|
| referência | distribuição $t_{n-2}$ deduzida do modelo | distribuição das estimativas reamostradas |
| quando usar | modelo linear plausível, linhas independentes e variância aproximadamente constante | distribuição amostral incerta, normalidade questionável ou estimador complexo, com linhas independentes e representativas |
| amostra pequena | fornece inferência exata se os erros condicionais forem normais | pode reproduzir mal a população quando há poucas observações ou poucos valores distintos |
| amostra grande | rápido e geralmente robusto a desvios moderados de normalidade | útil como verificação de sensibilidade, mas pode acrescentar pouco quando a aproximação teórica já é boa |
| heteroscedasticidade | o erro-padrão clássico pode estar errado; considere erro-padrão robusto | o bootstrap por pares preserva a heteroscedasticidade observada, desde que os pares sejam independentes |
| dependência | exige erro-padrão ou modelo apropriado para a dependência | exige bootstrap em blocos, por clusters ou outra reamostragem compatível |
| caudas muito extremas | calcula valores-$p$ muito pequenos diretamente | a resolução é limitada por $B$; estimar caudas raras exige muitas réplicas |
| custo | praticamente imediato | requer refazer o ajuste muitas vezes |

**Vantagens do teste $t$.** Ele é rápido, transparente e fornece
valores-$p$ e intervalos com resolução analítica. Quando o modelo
gaussiano homoscedástico é adequado, também é eficiente. Com amostras
grandes, a distribuição $t$ se aproxima da Normal e desvios moderados de
normalidade costumam ser menos problemáticos.

**Desvantagens do teste $t$.** A versão clássica depende do erro-padrão
correto. Heteroscedasticidade, dependência, outliers influentes e uma
função média mal especificada podem tornar a inferência enganosa. Nesses
casos, simplesmente ter uma amostra grande não corrige todos os
problemas.

**Vantagens do bootstrap.** Ele evita escolher uma forma paramétrica
específica para a distribuição amostral e pode representar assimetria,
heteroscedasticidade e estimadores para os quais o erro-padrão analítico
é difícil. O bootstrap por pares mantém juntas as variáveis de cada
pessoa e, assim, preserva a estrutura empírica entre idade e despesas.

**Desvantagens do bootstrap.** A população provisória é apenas a amostra
observada. Se essa amostra for pequena, enviesada ou não representar
regiões importantes da população, o bootstrap repetirá o problema. Ele
também não elimina confundimento nem corrige uma função média
inadequada. Para séries temporais, medidas pareadas ou dados agrupados,
reamostrar linhas isoladas quebra a dependência e é necessário usar
blocos, pares ou clusters. Finalmente, um teste bootstrap precisa impor
$H_0$ — por recentralização ou ajuste do modelo nulo — e valores-$p$
muito pequenos exigem um número enorme de réplicas.

> **Decisão prática para este exemplo**
>
> Com 1.338 observações, o teste $t$ e a aproximação Normal são
> praticamente iguais. O bootstrap por pares funciona como uma análise
> de sensibilidade e produz erro-padrão e intervalo muito próximos. A
> concordância aumenta a confiança na conclusão sobre a inclinação, mas
> o diagnóstico ainda mostra que idade sozinha é um modelo incompleto:
> antes de preferir qualquer valor-$p$, devemos incluir tabagismo e
> revisar a função média.

### Etapa 4: análise dos resíduos

O teste e o intervalo foram calculados dentro do modelo especificado.
Agora investigamos se esse modelo representa adequadamente os dados.
Cada diagnóstico responde a uma pergunta diferente:

1.  a função média deixa algum padrão sistemático sem explicar?
2.  a forma dos resíduos é aproximadamente compatível com uma Normal?
3.  a magnitude dos resíduos permanece aproximadamente constante?

Usaremos o próprio resíduo bruto

$$e_i=y_i-\widehat y_i,$$

medido em dólares. Isso mantém a leitura dos gráficos ligada diretamente
à pergunta: quanto a despesa observada ficou acima ou abaixo da
previsão?

#### Diagnóstico 1: resíduos contra valores ajustados

No eixo horizontal colocamos $\widehat y_i$, a despesa prevista usando
apenas a idade. No eixo vertical colocamos $e_i=y_i-\widehat y_i$. Um
modelo adequado deveria produzir uma nuvem sem forma reconhecível,
equilibrada ao redor de zero e com dispersão semelhante ao longo do eixo
horizontal.

In [ ]:
residuos = ajuste.resid
diagnostico = pd.DataFrame({
    "ajustado": ajuste.fittedvalues,
    "resíduo": residuos,
    "fumante": df["smoker"],
})

plt.figure(figsize=(9, 5))
sns.scatterplot(
    data=diagnostico,
    x="ajustado",
    y="resíduo",
    hue="fumante",
    alpha=.50,
)
plt.axhline(0, color="black", linestyle="--")
plt.xlabel("valor ajustado (US$)")
plt.ylabel("resíduo (US$)")
plt.show()

In [ ]:
diagnostico.groupby("fumante")["resíduo"].agg(
    quantidade="count",
    média="mean",
    desvio_padrão="std",
).round(2)

> **Como ler o resultado**
>
> Os resíduos de todos os indivíduos somam zero por construção, mas isso
> não garante média zero dentro de cada grupo. Não fumantes apresentam
> resíduo médio de aproximadamente $-4.882$ US\$: o modelo tende a
> superestimar suas despesas. Fumantes apresentam resíduo médio próximo
> de $18.958$ US\$: o modelo tende a subestimá-las. Os patamares revelam
> que `smoker` está ausente da função média. Esse é um problema mais
> fundamental do que uma simples falta de normalidade.

#### Diagnóstico 2: gráfico Q–Q

O gráfico Q–Q ordena os resíduos, $e_{(1)}\le\cdots\le e_{(n)}$, e
compara suas posições com os quantis esperados de uma distribuição
Normal:

$$
q_i=\Phi^{-1}\!\left(\frac{i-0{,}5}{n}\right).
$$

Se os resíduos fossem aproximadamente normais, os pares $(q_i,e_{(i)})$
ficariam próximos de uma reta. Curvatura sistemática indica diferenças
de assimetria ou caudas; pontos isolados muito afastados podem indicar
observações extremas.

In [ ]:
sm.qqplot(residuos, line="s", alpha=.45)
plt.xlabel("quantis teóricos da Normal")
plt.ylabel("resíduos ordenados (US$)")
plt.show()

> **Como ler o resultado**
>
> Os pontos observados formam uma curva pronunciada, com cauda superior
> muito acima da referência. A distribuição não se parece com uma única
> Normal simétrica. Parte dessa forma vem da mistura de fumantes e não
> fumantes, que possuem médias e dispersões distintas. O Q–Q avalia a
> **forma distributiva**; ele não verifica sozinho linearidade,
> independência ou causalidade.

#### Diagnóstico 3: o tamanho dos erros muda?

Para tornar a pergunta mais direta, colocamos a magnitude do erro

$$|e_i|=|y_i-\widehat y_i|$$

contra o valor ajustado. O valor absoluto remove o sinal: um erro de
$-5.000$ US\$ e um erro de $5.000$ US\$ têm o mesmo tamanho. Se a
variabilidade for aproximadamente constante, a tendência da magnitude
deve permanecer em uma altura semelhante ao longo do eixo horizontal.

In [ ]:
diagnostico["magnitude"] = np.abs(diagnostico["resíduo"])

plt.figure(figsize=(9, 5))
sns.scatterplot(
    data=diagnostico,
    x="ajustado",
    y="magnitude",
    hue="fumante",
    alpha=.35,
)

for grupo, dados_grupo in diagnostico.groupby("fumante"):
    faixas = pd.qcut(dados_grupo["ajustado"], q=8, duplicates="drop")
    tendencia = dados_grupo.groupby(faixas, observed=True)[["ajustado", "magnitude"]].mean()
    plt.plot(tendencia["ajustado"], tendencia["magnitude"], linewidth=3, label=f"tendência: {grupo}")

plt.xlabel("valor ajustado (US$)")
plt.ylabel("erro absoluto (US$)")
plt.legend()
plt.show()

> **Como ler o resultado**
>
> O aspecto dominante é a separação entre grupos: fumantes mantêm erros
> absolutos sistematicamente maiores. Assim, uma única variância
> $\sigma^2$ para toda a amostra é questionável. Antes de escolher uma
> correção de erro-padrão, devemos incluir tabagismo na média e refazer
> os diagnósticos; parte da heterogeneidade pode ser consequência direta
> da variável omitida.

> **Ordem recomendada do diagnóstico**
>
> Revise primeiro a função média, depois examine novamente forma e
> variância dos resíduos. Aplicar bootstrap ou erros-padrão robustos a
> uma média claramente mal especificada melhora a estimativa da
> incerteza de um modelo que continua descrevendo inadequadamente os
> dados.

### Etapa 5: conclusão responsável

O exemplo produz uma estimativa clara, um intervalo e um teste, mas
termina com uma revisão do próprio modelo:

1.  a associação média estimada é positiva: $\widehat\beta_1=257{,}72$;
2.  o IC de 95% é $[213{,}58;301{,}87]$;
3.  tanto o teste $t$ quanto o bootstrap rejeitam $H_0:\beta_1=0$;
4.  idade isolada explica pouco da variabilidade individual;
5.  tabagismo deve entrar na função média antes de usar o modelo para
    explicação ou previsão;
6.  nenhuma dessas contas, por si só, identifica um efeito causal.

> **Ideia central**
>
> Uma regressão completa não termina quando o software imprime
> coeficientes e valores-$p$. Ajuste, incerteza, teste e diagnóstico
> respondem a perguntas complementares e precisam ser interpretados em
> conjunto.

## Da reta à verossimilhança: o caminho completo

Nesta aula, acrescentamos uma camada probabilística à regressão da Aula
17:

1.  **A reta fornece a média.** Para cada idade $x_i$, calculamos
    $\mu_i=\beta_0+\beta_1x_i$.
2.  **Os indivíduos não ficam exatamente sobre a média.** Representamos
    a diferença por um erro $\varepsilon_i$.
3.  **Escolhemos uma distribuição para esses erros.** A Normal descreve
    erros simétricos em torno de zero, com escala controlada por
    $\sigma$.
4.  **Cada reta candidata gera densidades diferentes para os dados.**
    Uma reta distante dos pontos atribui densidade pequena às
    observações.
5.  **Multiplicamos as contribuições.** Sob independência, obtemos a
    verossimilhança conjunta.
6.  **Maximizamos.** Com erros normais e variância constante, isso
    equivale a minimizar a soma dos quadrados dos resíduos.

Essa equivalência explica por que mínimos quadrados reaparece. O
quadrado está no expoente da densidade Normal; ao tomar o log, ele se
transforma justamente na soma de quadrados.

## Os resíduos precisam parecer perfeitamente normais?

Não. A reta pode ser calculada mesmo quando a distribuição dos resíduos
não é Normal. A normalidade é mais importante para intervalos e testes
clássicos em amostras pequenas.

Na prática, olhamos primeiro para problemas mais estruturais:

-   tendência curva nos resíduos: a média não foi bem representada;
-   formato de funil: a variabilidade muda com o valor previsto;
-   grupos separados: pode faltar uma variável importante;
-   poucos resíduos extremos: algumas observações podem dominar o
    ajuste;
-   dependência temporal ou por pessoa: a informação efetiva é menor do
    que o número de linhas sugere.

O QQ-plot é uma parte desse diagnóstico, não um teste que aprova ou
reprova sozinho toda a regressão.

> **Ideia central**
>
> Mínimos quadrados diz como escolher a reta. A hipótese Normal explica
> como transformar a distância dos pontos à reta em uma verossimilhança
> e em procedimentos de inferência.

## Questões de revisão

1.  Derive a log-verossimilhança gaussiana a partir do produto.
2.  Identifique os termos constantes em $\beta_0,\beta_1$.
3.  Por que maximizar $-\operatorname{SSE}/(2\sigma^2)$ equivale a
    minimizar SSE?
4.  Diferencie erro normal, resíduo aparentemente normal e média
    condicional correta.
5.  Por que o padrão por tabagismo sugere revisar a média, não apenas a
    variância?

## Bibliografia

-   James et al., *An Introduction to Statistical Learning*, capítulo 3.
-   Wasserman, *All of Statistics*, MLE e regressão.
-   Faraway, *Linear Models with R*, capítulos 2–4.
-   Gelman, Hill e Vehtari, *Regression and Other Stories*, capítulos
    6–11.
-   Fox, *Applied Regression Analysis and Generalized Linear Models*,
    diagnóstico.